# tf.data によるデータの読み込みと前処理の最適化

**学習目標**
1. tf.data を使用してメモリからデータを読み込む方法を学ぶ
1. tf.data を使用してディスクからデータを読み込む方法を学ぶ
1. 特徴量エンジニアリング（バッチ処理、シャッフルなど）を含む本番用入力パイプラインを作成する方法を学ぶ
1. tf.data を使用してパイプラインを最適化する方法を学ぶ


このノートブックでは、まず前のラボで実装した線形回帰をリファクタリングして `tf.data.Dataset` からデータを取得するようにし、それを使って**確率的勾配降下法（SGD）**を実装する方法を学びます。このケースでは、元のデータセットは合成データであり、`tf.data` API によってメモリから直接読み込まれます。

フレームワークには TensorFlow を使用しますが、**tf.data は JAX や PyTorch など任意のフレームワークでも機能します**。

後半では、データセットがディスク上にある場合に `tf.data` API を使用してデータセットを読み込む方法を学び、データパイプラインを最適化する方法について学びます。

In [ ]:
import os
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

import json
import math
from pprint import pprint

import numpy as np
import tensorflow as tf

print(tf.version.VERSION)

## メモリからのデータ読み込み

### データセットの作成

前のセクションの合成データセットを考えてみましょう。

In [ ]:
N_POINTS = 10
X = tf.constant(range(N_POINTS), dtype=tf.float32)
Y = 2 * X + 10

まず、以下を入力として受け取り、`tf.data.Dataset` を返す関数を実装します。

- 線形関数 $y= 2x + 10$ によって生成された合成データのベクトル $X$ および $Y$
- 訓練でデータセットを反復処理する回数（`epochs`）
- データセット内のバッチサイズ（`batch_size`）

**注記:** データセットを使い果たしたため、最後のバッチには指定した正確な要素数が含まれない可能性があることに注意してください。

各バッチで完全に同じ要素数を持つバッチが必要な場合は、以下のように設定して最後のバッチを破棄する必要があります。

```python
dataset = dataset.batch(batch_size, drop_remainder=True)
```

ここではその設定を行います。

**演習 1**: `create_dataset` 関数を実装してください。<br>
この関数は、入力テンソル `X` と `Y` から `tf.data.Dataset` を作成し、指定されたエポック数繰り返すように設定して、指定された `batch_size` のバッチを作成する必要があります。最後のバッチがバッチサイズより小さい場合は破棄（drop）してください。

In [ ]:
def create_dataset(X, Y, epochs, batch_size):
    dataset = tf.data.Dataset.from_tensor_slices((X, Y))
    # TODO: ここにコードを記述してください
    return dataset

3 つのデータポイントのバッチでデータセットを 2 回反復処理して、関数をテストしてみましょう。

In [ ]:
BATCH_SIZE = 3
EPOCH = 2

dataset = create_dataset(X, Y, epochs=EPOCH, batch_size=BATCH_SIZE)

for i, (x, y) in enumerate(dataset):
    print("x:", x.numpy(), "y:", y.numpy())
    assert len(x) == BATCH_SIZE
    assert len(y) == BATCH_SIZE
assert EPOCH

### 損失関数と勾配

損失関数と勾配を計算する関数は前と同じです。

In [ ]:
def loss_mse(X, Y, w0, w1):
    Y_hat = w0 * X + w1
    errors = (Y_hat - Y) ** 2
    return tf.reduce_mean(errors)


def compute_gradients(X, Y, w0, w1):
    with tf.GradientTape() as tape:
        loss = loss_mse(X, Y, w0, w1)
    return tape.gradient(loss, [w0, w1])

### 訓練ループ

以前との主な違いは、訓練ループ内で `create_dataset` 関数によって生成された `tf.data.Dataset` を直接反復処理する点です。

バッチサイズ 2 で合成データセットを 250 回反復するようにデータセットを設定します。

**演習 2**: 訓練ループを実装してください。データセットの各バッチに対して、以下を行う必要があります。
1. 重み `w0` および `w1` に対する損失の勾配を計算する。
2. 勾配と学習率を使用して重みを更新する。
3. 100 ステップごとに損失を計算し、ステップ番号、損失、および `w0` と `w1` の現在値を出力する。

最後の assert でモデルのパフォーマンスをチェックできるよう、ループ内で `loss` 変数を確実に更新してください。

In [ ]:
EPOCHS = 250
BATCH_SIZE = 2
LEARNING_RATE = 0.02

MSG = "STEP {step} - loss: {loss}, w0: {w0}, w1: {w1}\n"

w0 = tf.Variable(0.0)
w1 = tf.Variable(0.0)

dataset = create_dataset(X, Y, epochs=EPOCHS, batch_size=BATCH_SIZE)

for step, (X_batch, Y_batch) in enumerate(dataset):
    # TODO: ここにコードを記述してください
    pass

assert loss < 0.0001
assert abs(w0 - 2) < 0.001
assert abs(w1 - 10) < 0.001

## ディスクからのデータ読み込み

### CSV ファイルの配置確認

前のラボで書き出した **taxifare データセット** の CSV ファイルから始めます。

taxifare データセットのファイルは `../data` に保存されています。

下のセルを実行してファイルが存在することを確認し、存在しない場合は前のラボのノートブックを実行して taxifare データセットを再生成してください。

In [ ]:
!ls -l ../data/taxi*.csv

### 低レベル tf.data API を使用した CSV ファイルの読み込み

より柔軟なパイプラインを構築するために、低レベルの tf.data API を利用してパイプラインの動作を完全に制御できます。

CSV などのテキストベースのデータには、`TextLineDataset` を使用してデータを読み込むことができます。

In [ ]:
ds = tf.data.TextLineDataset("../data/taxi-train.csv")
ds

Dataset オブジェクト（`ds`）はまだ単なる定義にすぎず、実際のデータはまだ読み込まれていないことに注意してください。<br>
`dataset.take(2)` を使用して、このデータセットの最初の 2 つの要素を反復処理してみましょう。

In [ ]:
for data in ds.take(2):
    print(data)

ヘッダー行が最初の要素として読み込まれているようです。これは訓練データの一部ではないため、`skip()` メソッドでスキップしましょう。

In [ ]:
ds = tf.data.TextLineDataset("../data/taxi-train.csv").skip(1)

for data in ds.take(2):
    print(data)

### `.map()` による特徴量の変換

この時点では、CSV ファイルをテキストファイルとして読み込んだだけであり、各行は区切り文字（`,`）を含む単一の文字列値として表現されているにすぎません。

行を受け取り、複数の値に分割する解析関数を作成しましょう。

**演習 3**: 単一の CSV 行文字列を文字列のリストに分割する `parse_csv` 関数を実装してください。

In [ ]:
def parse_csv(row):
    # TODO: ここにコードを記述してください
    return ...

for ループで `.take()` を使用してこの関数を呼び出し、正常に動作することを確認しましょう。

In [ ]:
ds = tf.data.TextLineDataset("../data/taxi-train.csv").skip(1)

for data in ds.take(2):
    values = parse_csv(data)
    pprint(values)

for ループ内で関数を呼び出す代わりに、`.map()` メソッドでラップしてパイプラインに組み込むことができます。

In [ ]:
ds = tf.data.TextLineDataset("../data/taxi-train.csv").skip(1).map(parse_csv)

for data in ds.take(2):
    print(data)

次に、`parse_csv` 関数を拡張してみましょう。<br>
機械学習の訓練では、訓練データを `(features, label)` のタプルとして渡すのが一般的です。

この CSV ファイルには次の列があります。

In [ ]:
!head -1 ../data/taxi-train.csv

`pickuplon`、`pickuplat`、`dropofflon`、`dropofflat` を特徴量として使用し、`fare_amount` の値を予測したいとします。その場合は次のようになります。

**演習 4**: `(features, label)` のタプルを返すように `parse_csv` 関数を更新してください。<br>
ラベルは最初の列（`fare_amount`）とし、特徴量は 3 番目から 6 番目の列（`pickuplon`、`pickuplat`、`dropofflon`、`dropofflat`）とします。
抽出した値を数値に変換することを忘れないでください。

In [ ]:
def parse_csv(row):
    columns = tf.strings.split(row, ",")
    # ラベル: fare_amount
    label = ...  # TODO: ここにコードを記述してください
    # 特徴量: pickup_longitude, pickup_latitude, dropoff_longitude, dropoff_latitude
    features = ...  # TODO: ここにコードを記述してください
    return features, label

In [ ]:
ds = tf.data.TextLineDataset("../data/taxi-train.csv").skip(1)
ds = ds.map(parse_csv)

for features, label in ds.take(2):
    print(f"features: \n  {features}, \nlabel: \n  {label} \n++++")

### バッチ処理

通常、機械学習の訓練モジュールにはバッチ処理されたデータが必要です。`.batch(BATCH_SIZE)` を追加してデータをバッチ処理するようにパイプラインをリファクタリングしましょう。

In [ ]:
BATCH_SIZE = 4

ds = tf.data.TextLineDataset("../data/taxi-train.csv").skip(1)
ds = ds.map(parse_csv).batch(BATCH_SIZE)

for features, label in ds.take(2):
    print(f"features: \n  {features}, \nlabel: \n  {label} \n++++")

これで、データセットは*行*ではなく*バッチ*のイテレータになり、ニューラルネットワークのミニバッチ訓練に適した形式になりました。

### シャッフルとリピート

複数のワーカーでバッチ処理を行ってディープラーニングモデルを訓練する場合、データをシャッフルすると効果的です。これにより、異なるワーカーが同時に入力ファイルの異なる部分を処理することになり、ワーカー間で勾配を平均化する際に役立ちます。<br>
`.shuffle()` を使用してシャッフルを追加できます。ただし、`buffer_size` で指定されたシャッフルバッファはメモリ上に保存されるため、非常に大規模なデータセットの完全なシャッフルには適していないことに注意してください。

データパイプラインを `create_dataset` 関数でラップして、その動作を制御し、データセットが訓練に使用される場合にのみデータをシャッフルできるようにしましょう。

関数本体がデータをシャッフルする必要がある場合（`mode == "train"`）と、シャッフルすべきではない場合（`mode == "eval"`）を区別できるように、関数に追加の引数 `mode` を導入します。

また、訓練中にデータを無期限に読み込めるよう、`.repeat()` を追加します。

**演習 5**: ファイルパターンを受け取る `create_dataset` 関数を実装してください。この関数は以下を行う必要があります。
1. ファイルパターンから `TextLineDataset` を作成し、ヘッダー行をスキップする。
2. 各行に `parse_csv` 関数をマッピングする。
3. データセットが無期限に繰り返す（repeat）ようにする。
4. mode が 'train' の場合はデータセットをシャッフルする。
5. データセットをバッチ処理し、端数（remainder）を破棄する。

In [ ]:
def create_dataset(pattern, batch_size, mode="eval"):
    # TODO: ここにコードを記述してください
    pass

両方のモードで関数が正常に機能することを確認しましょう。

In [ ]:
# このセルを複数回実行して、結果が異なることを確認します。
tempds = create_dataset("../data/taxi-train.csv", 2, "train")
print(list(tempds.take(1)))

In [ ]:
tempds = create_dataset("../data/taxi-valid.csv", 2, "eval")
print(list(tempds.take(1)))

## tf.data によるパフォーマンスの向上

データの読み込みと前処理フェーズのパフォーマンスを最大化することは、多くの機械学習のユースケースで極めて重要です。

`tf.data` は、パフォーマンスのボトルネックの原因に応じて、プロセスを最適化するためのさまざまな方法を提供します。<br>
いくつかのシナリオを見てみましょう。

比較のために、訓練アプリケーションループをシミュレートするこの `benchmark` 関数を使用します。

In [ ]:
import time


def benchmark(dataset, num_epochs=2):
    start_time = time.perf_counter()
    for epoch_num in range(num_epochs):
        for sample in dataset:
            # 訓練ステップを実行
            time.sleep(0.01)
    print("Execution time:", time.perf_counter() - start_time)

### ケース 1: 負荷の高い map 処理によるパフォーマンスのボトルネック

特徴量変換の `.map()` は柔軟で便利ですが、前処理関数に負荷の高い処理が含まれている場合、この処理がパフォーマンスのボトルネックになる可能性があります。

解析関数にスリープ時間を追加して、そのケースをシミュレートしてみましょう。

In [ ]:
def heavy_parse_csv(row):
    columns = tf.strings.split(row, ",")
    label = tf.strings.to_number(columns[0])
    features = tf.strings.to_number(columns[2:6])

    # 負荷の高い前処理を実行...
    tf.py_function(lambda: time.sleep(0.001), [], ())

    return features, label

In [ ]:
def create_dataset(pattern, batch_size=128):
    ds = tf.data.TextLineDataset(pattern).skip(1)
    ds = ds.map(heavy_parse_csv)
    ds = ds.batch(batch_size, drop_remainder=True)
    return ds

In [ ]:
tempds = create_dataset("../data/taxi-train.csv")
benchmark(tempds)

フローは次のようになります。この場合、データの読み込みと訓練の間の map 処理がボトルネックになっています。

![Map bottleneck](https://www.tensorflow.org/guide/images/data_performance/sequential_map.svg)

このプロセスをどのように最適化できるか見てみましょう。

#### 解決策 1: map の並列化

入力要素は互いに独立しているため、前処理を複数の CPU コアにわたって並列化できます。これを可能にするため、map 変換には並列処理のレベルを指定する num_parallel_calls 引数が用意されています。

`.map()` では、関数と共に `num_parallel_calls` 引数を指定できます。`tf.data.AUTOTUNE` を指定することで、並列度を自動調整できます。

In [ ]:
def create_dataset(pattern, batch_size=128):
    ds = tf.data.TextLineDataset(pattern).skip(1)
    ds = ds.map(heavy_parse_csv, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size, drop_remainder=True)
    return ds

In [ ]:
tempds = create_dataset("../data/taxi-train.csv")
benchmark(tempds)

はるかに高速になりました。

現在のフローは次のようになります。
![parallelized](https://www.tensorflow.org/guide/images/data_performance/parallel_map.svg)

#### 解決策 2: map 処理のベクトル化
`.map()` は `Dataset` から返される個々の要素を処理します。現在の関数は、一度に 1 つの CSV 要素に対して動作するように構造化されています。しかし、可能な場合はバッチ単位でデータを処理する方が常に効率的です。

関数をベクトル化し（つまり、一度に入力のバッチ全体に対して動作するようにし）、`map` 変換の前に `batch` 変換を適用してみましょう。

In [ ]:
def heavy_parse_csv_batch(row):
    columns = tf.strings.split(row, ",").to_tensor()
    label = tf.strings.to_number(columns[:, 0])
    features = tf.strings.to_number(columns[:, 2:6])

    # 負荷の高い前処理を実行...
    tf.py_function(lambda: time.sleep(0.001), [], ())

    return features, label

In [ ]:
def create_dataset(pattern, batch_size=128):
    ds = tf.data.TextLineDataset(pattern).skip(1)
    ds = ds.batch(batch_size, drop_remainder=True)
    ds = ds.map(heavy_parse_csv_batch, num_parallel_calls=tf.data.AUTOTUNE)
    return ds

In [ ]:
tempds = create_dataset("../data/taxi-train.csv")
benchmark(tempds)

### ケース 2: I/O（データ読み込み）におけるパフォーマンスのボトルネック
次のシナリオを見てみましょう。

実際の運用環境では、入力データがリモート（例えば別のロケーションの Google Cloud Storage など）に保存されている場合があります。ローカルでデータを読み込む際には適切に機能するデータセットパイプラインであっても、ローカルストレージとリモートストレージの間にある次のような違いにより、リモートからデータを読み込む際に I/O がボトルネックになる可能性があります。

- **Time-to-first-byte（最初の 1 バイトを受信するまでの時間）**: リモートストレージからファイルの最初の 1 バイトを読み込むには、ローカルストレージよりも桁違いに長い時間がかかることがあります。
- **読み込みスループット**: リモートストレージは通常、大きな総帯域幅を提供しますが、単一のファイルを読み込む場合はこの帯域幅のごく一部しか利用できないことがあります。

さらに、生のバイトデータがメモリに読み込まれた後、データのデシリアライズや復号（例: protobuf）が必要になる場合もあり、これには追加の計算が必要です。このオーバーヘッドはデータがローカルまたはリモートのどちらに保存されているかに関係なく発生しますが、データが効果的にプリフェッチされていない場合、リモート環境ではさらに悪化する可能性があります。

このシナリオをシミュレートするカスタムデータセットを作成してみましょう。

In [ ]:
class IOBoundDataset(tf.data.Dataset):
    def _generator(file_name, num_samples):
        # ファイルを開く
        time.sleep(0.3)

        for sample_idx in range(num_samples):
            # ファイルから各行を読み込む
            time.sleep(0.15)

            yield (sample_idx,)

    def __new__(cls, file_name, num_samples=5):
        return tf.data.Dataset.from_generator(
            cls._generator,
            output_signature=tf.TensorSpec(shape=(1,), dtype=tf.int64),
            args=(
                file_name,
                num_samples,
            ),
        )

In [ ]:
ds = IOBoundDataset("dummy_file.csv").repeat(20)
benchmark(ds)

#### 解決策 1: キャッシュ
機械学習の訓練では同じデータセットを繰り返し使用することが多いため、毎回リモートソースから再読み込みするのではなく、最初のエポックの間にデータをキャッシュし、それ以降のエポックではキャッシュからデータを取得するのが優れた戦略です。

`.cache()` を挿入するだけでキャッシュを使用できます。

In [ ]:
ds = IOBoundDataset("dummy_file.csv").cache().repeat(20)
benchmark(ds)

はるかに高速になったようです。ただし、キャッシュされたデータはメモリ上に保存されるため、何をキャッシュするかには注意してください。例えば、テラバイト規模のデータセットをキャッシュするのは現実的ではありません。

例えば、訓練用の動画のパスが含まれる CSV ファイルがある場合、実際の動画ファイルをキャッシュする代わりに、動画読み込み関数の前に `cache()` を挿入して小さな CSV ファイルをキャッシュすることを検討してください。

```python 
dataset.map(parse_csv_fn).cache().map(load_video_fn)
```

#### 解決策 2: インターリーブ（Interleave）

さまざまなデータ抽出のオーバーヘッドの影響を軽減するために、tf.data.Dataset.interleave 変換を使用してデータ読み込みステップを並列化し、他のデータセット（データファイルリーダーなど）の内容をインターリーブ（交互に挟み込み）できます。

複数のシャード化されたファイルがあるとしましょう。

In [ ]:
files = [
    "dummy_file_shard001.csv",
    "dummy_file_shard002.csv",
    "dummy_file_shard003.csv",
]

`interleave()` を挿入して、複数のファイル読み込み処理をインターリーブできます。

In [ ]:
ds = tf.data.Dataset.from_tensor_slices(files)
ds = ds.interleave(IOBoundDataset)

benchmark(ds)

どのように読み込まれるか確認してみましょう。

In [ ]:
for d in ds:
    print(d)

各データセットには 0 から 4 までの値が含まれており、ここでは 3 つのファイルからのデータ読み込みがインターリーブされていることが確認できます。

これはインターリーブのフロー図です（この図では 2 つのファイルの場合）。
![sequential interleave](https://www.tensorflow.org/guide/images/data_performance/sequential_interleave.svg)

`.map()` と同様に、`num_parallel_calls` を追加してこのインターリーブ処理を並列化することで、パフォーマンスをさらに向上させることができます。

In [ ]:
ds = tf.data.Dataset.from_tensor_slices(files)
ds = ds.interleave(IOBoundDataset, num_parallel_calls=len(files))

benchmark(ds)

![parallel interleave](https://www.tensorflow.org/guide/images/data_performance/parallel_interleave.svg)

これらの手法を組み合わせることで、高度に最適化されたデータ読み込みと変換のパイプラインを設計できます。

Copyright 2025 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

     https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.